In [1]:
import requests
import zipfile
import io
from pathlib import Path

GEO_DIR = Path('../data/geology')
GEO_DIR.mkdir(exist_ok=True)

# USGS NGMDB - Nevada state geology (1:500k)
url = 'https://mrdata.usgs.gov/geology/state/shp/NV.zip'

print('Downloading Nevada geology shapefile...')
resp = requests.get(url, timeout=120, stream=True)
resp.raise_for_status()

downloaded = 0
chunks = []
for chunk in resp.iter_content(chunk_size=1024*1024):
    chunks.append(chunk)
    downloaded += len(chunk)
    print(f'  {downloaded/1e6:.1f} MB', end='\r')

print(f'\nDownload complete: {downloaded/1e6:.1f} MB')

# Extract
z = zipfile.ZipFile(io.BytesIO(b''.join(chunks)))
z.extractall(GEO_DIR)
print(f'Extracted to: {GEO_DIR}')
print('Files:')
for f in GEO_DIR.iterdir():
    print(f'  {f.name}')

  69.1 MB
Download complete: 69.1 MB
Extracted to: ..\data\geology
Files:
  NV_geol_poly.dbf
  NV_geol_poly.prj
  NV_geol_poly.shp
  NV_geol_poly.shx
  NV_lith.csv
  NV_ref.csv
  NV_structure.dbf
  NV_structure.prj
  NV_structure.shp
  NV_structure.shx
  NV_units.csv


In [2]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

GEO_DIR = Path('../data/geology')

# Load geology polygons
gdf = gpd.read_file(GEO_DIR / 'NV_geol_poly.shp')
print(f'Total geology polygons: {len(gdf)}')
print(f'Columns: {list(gdf.columns)}')
print()

# Load units lookup
units = pd.read_csv(GEO_DIR / 'NV_units.csv', encoding='latin1')
print(f'Units table columns: {list(units.columns)}')
print()

# Show sample of unit codes
print('Sample units:')
print(units.head(20).to_string())

Total geology polygons: 30763
Columns: ['STATE', 'ORIG_LABEL', 'SGMC_LABEL', 'UNIT_LINK', 'REF_ID', 'GENERALIZE', 'SRC_URL', 'URL', 'geometry']

Units table columns: ['state', 'orig_label', 'map_sym1', 'map_sym2', 'unit_link', 'prov_no', 'province', 'unit_name', 'unit_age', 'unitdesc', 'strat_unit', 'unit_com', 'map_ref', 'rocktype1', 'rocktype2', 'rocktype3', 'unit_ref', 'map_symb2']

Sample units:
   state orig_label map_sym1 map_sym2  unit_link  prov_no  province                                                                                  unit_name                            unit_age                                                                                                                                                                                                                                                                                                                 unitdesc                                                                                          

In [3]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

GEO_DIR = Path('../data/geology')

# Load geology polygons and units table
gdf   = gpd.read_file(GEO_DIR / 'NV_geol_poly.shp')
units = pd.read_csv(GEO_DIR / 'NV_units.csv', encoding='latin1')

# Join units to polygons
gdf = gdf.merge(units, left_on='UNIT_LINK', right_on='unit_link', how='left')

# Find all Triassic units
triassic = gdf[gdf['unit_age'].str.contains('Triassic', na=False)].copy()
print(f'Total Triassic polygons: {len(triassic)}')
print(f'\nUnique Triassic units:')
print(triassic[['ORIG_LABEL','unit_name','unit_age','rocktype1']].drop_duplicates().to_string())

# Filter to marine units — limestone, dolomite, chert, shale (not volcanic/intrusive)
marine_keywords = ['limestone','dolostone','dolomite','chert','shale','siltstone','mudstone','carbonate']
marine_mask = triassic['rocktype1'].str.lower().str.contains(
    '|'.join(marine_keywords), na=False)
triassic_marine = triassic[marine_mask].copy()
print(f'\nTriassic marine polygons: {len(triassic_marine)}')

# Save
out_path = GEO_DIR / 'triassic_marine.shp'
triassic_marine.to_file(out_path)
print(f'Saved: {out_path}')

Total Triassic polygons: 346

Unique Triassic units:
      ORIG_LABEL                                                                                     unit_name                           unit_age  rocktype1
5016        JTRs               Shale, mudstone, siltstone, sandstone, and carbonate rock; sparse volcanic rock    Late Triassic to Early Jurassic  claystone
23101        TRc  Limestone, minor amounts of dolomite, shale, and sandstone; locally thick conglomerate units                           Triassic  limestone
23573       TRmt                                      Moenkopi Formation, Thaynes Formation, and related rocks  Early Triassic to Middle Triassic      shale

Triassic marine polygons: 312
Saved: ..\data\geology\triassic_marine.shp


C:\Users\brook\Documents\Project-PaleoWave\.pixi\envs\default\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'state' to 'state_1'
  ogr_write(
C:\Users\brook\Documents\Project-PaleoWave\.pixi\envs\default\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'orig_label' to 'orig_lab_1'
  ogr_write(
C:\Users\brook\Documents\Project-PaleoWave\.pixi\envs\default\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'unit_link' to 'unit_lin_1'
  ogr_write(
C:\Users\brook\Documents\Project-PaleoWave\.pixi\envs\default\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Value 'LIMESTONE, MINOR AMOUNTS OF DOLOMITE, SHALE, AND SANDSTONE; LOCALLY THICK CONGLOMERATE UNITS (Lower, Middle, and Upper Triassic)-Includes Tobin, Dixie Valley, Favret, Augusta Mountain, and Cane Spring Formations and Star Peak Group in central Nevada and Grantsville and Luning Formations in west-central Nevada' o

In [4]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
from shapely.geometry import Point

DATA_DIR  = Path('../data')
GEO_DIR   = DATA_DIR / 'geology'
MODEL_DIR = DATA_DIR / 'model'

# Load top candidates
candidates = gpd.read_file(MODEL_DIR / 'top_candidates.geojson')
print(f'Candidates: {len(candidates)}')

# Load Triassic marine geology
geology = gpd.read_file(GEO_DIR / 'triassic_marine.shp')
print(f'Triassic marine polygons: {len(geology)}')

# Make sure both are in the same CRS
geology = geology.to_crs(candidates.crs)

# Spatial join — find which candidates fall inside Triassic marine polygons
joined = gpd.sjoin(candidates, 
                   geology[['ORIG_LABEL','unit_name','unit_age','rocktype1','unitdesc','geometry']], 
                   how='left', 
                   predicate='within')

# Flag whether inside a Triassic formation
joined['on_triassic'] = ~joined['ORIG_LABEL'].isna()

# For candidates not within a polygon, find distance to nearest Triassic formation
print('Calculating distance to nearest Triassic formation...')
candidates_copy = candidates.to_crs('EPSG:32611')  # UTM zone 11N for Nevada — meters
geology_utm     = geology.to_crs('EPSG:32611')
geology_union   = geology_utm.union_all()

joined['dist_to_triassic_m'] = candidates_copy.geometry.apply(
    lambda g: round(g.distance(geology_union))
)

# Build final prioritized table
final = joined[[
    'rank', 'latitude', 'longitude', 'probability',
    'on_triassic', 'dist_to_triassic_m',
    'ORIG_LABEL', 'unit_name', 'rocktype1'
]].copy()

# Rename for clarity
final.columns = [
    'rank', 'latitude', 'longitude', 'ml_probability',
    'on_triassic', 'dist_to_triassic_m',
    'formation_code', 'formation_name', 'rock_type'
]

# Composite score: ML probability + geology bonus
# On Triassic = full score, within 500m = partial, further = penalty
def geo_bonus(row):
    if row['on_triassic']:
        return 0.20
    elif row['dist_to_triassic_m'] <= 500:
        return 0.10
    elif row['dist_to_triassic_m'] <= 2000:
        return 0.05
    else:
        return 0.0

final['geo_bonus']        = final.apply(geo_bonus, axis=1)
final['composite_score']  = (final['ml_probability'] + final['geo_bonus']).clip(0, 1).round(4)
final['priority']         = final['composite_score'].rank(ascending=False).astype(int)

# Sort by composite score
final = final.sort_values('composite_score', ascending=False).reset_index(drop=True)

# Save
out_path = MODEL_DIR / 'priority_targets.csv'
final.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')
print(f'\nTop 20 Priority Targets:')
print(final.head(20)[[
    'priority','latitude','longitude','ml_probability',
    'on_triassic','dist_to_triassic_m','formation_name','composite_score'
]].to_string(index=False))

Candidates: 50
Triassic marine polygons: 312
Calculating distance to nearest Triassic formation...

Saved: ..\data\model\priority_targets.csv

Top 20 Priority Targets:
 priority  latitude   longitude  ml_probability  on_triassic  dist_to_triassic_m formation_name  composite_score
        1 40.404722 -118.243889        0.800458        False                  53            NaN           0.9005
        2 40.787222 -117.456389        0.853346        False               13040            NaN           0.8533
        3 40.907222 -118.464444        0.852320        False               41459            NaN           0.8523
        4 40.209167 -117.587778        0.795162        False                 993            NaN           0.8452
        5 40.550278 -118.232500        0.794605        False                1361            NaN           0.8446
        6 40.434444 -117.686667        0.842316        False                4782            NaN           0.8423
        7 40.839722 -117.720833        0.

In [5]:
# Check CRS match and see what the join actually produced
print(f'Candidates CRS: {candidates.crs}')
print(f'Geology CRS:    {geology.crs}')

# How many candidates are within ANY geology polygon?
joined_all = gpd.sjoin(candidates, geology[['ORIG_LABEL','unit_name','geometry']], 
                       how='left', predicate='intersects')
print(f'\nCandidates intersecting Triassic polygons: {joined_all[~joined_all["ORIG_LABEL"].isna()]["rank"].nunique()}')

# Check the closest candidate (#1 at 53m)
target = candidates_copy[candidates_copy['rank'] == 1]
print(f'\nCandidate #1 geometry: {target.geometry.values[0]}')

# Find the nearest Triassic polygon
nearest = geology_utm.copy()
nearest['dist'] = nearest.geometry.distance(target.geometry.values[0])
nearest = nearest.sort_values('dist').head(3)
print(f'\nNearest Triassic polygons to candidate #1:')
print(nearest[['ORIG_LABEL','unit_name','dist']].to_string())

Candidates CRS: EPSG:4326
Geology CRS:    EPSG:4326

Candidates intersecting Triassic polygons: 0

Candidate #1 geometry: POINT (461493.765448732 4515237.331509801)

Nearest Triassic polygons to candidate #1:
   ORIG_LABEL                                                                                     unit_name          dist
32        TRc  Limestone, minor amounts of dolomite, shale, and sandstone; locally thick conglomerate units  13040.081910
2         TRc  Limestone, minor amounts of dolomite, shale, and sandstone; locally thick conglomerate units  14576.525100
82        TRc  Limestone, minor amounts of dolomite, shale, and sandstone; locally thick conglomerate units  16250.623765


In [6]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

DATA_DIR  = Path('../data')
GEO_DIR   = DATA_DIR / 'geology'
MODEL_DIR = DATA_DIR / 'model'

# Reload both clean
candidates = gpd.read_file(MODEL_DIR / 'top_candidates.geojson')
geology    = gpd.read_file(GEO_DIR / 'triassic_marine.shp')

# Fix any invalid geometries
geology['geometry'] = geology.geometry.buffer(0)
candidates['geometry'] = candidates.geometry.buffer(0)

print(f'Candidates CRS: {candidates.crs}')
print(f'Geology CRS:    {geology.crs}')
print(f'Geology valid:  {geology.is_valid.all()}')

# Reproject both to UTM 11N for accurate distance calculations
cands_utm = candidates.to_crs('EPSG:32611')
geo_utm   = geology.to_crs('EPSG:32611')

print(f'\nSample candidate coords (UTM):')
print(cands_utm.geometry.head(3))
print(f'\nSample geology bbox (UTM):')
print(geo_utm.total_bounds)

Candidates CRS: EPSG:4326
Geology CRS:    EPSG:4326
Geology valid:  True

Sample candidate coords (UTM):
0    POLYGON EMPTY
1    POLYGON EMPTY
2    POLYGON EMPTY
Name: geometry, dtype: geometry

Sample geology bbox (UTM):
[ 392934.5131625  3961817.20539613  764033.15918645 4653145.59172078]


In [7]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
from shapely.geometry import Point

DATA_DIR  = Path('../data')
GEO_DIR   = DATA_DIR / 'geology'
MODEL_DIR = DATA_DIR / 'model'

# Rebuild candidates from CSV (clean points)
df = pd.read_csv(MODEL_DIR / 'priority_targets.csv')
candidates = gpd.GeoDataFrame(
    df,
    geometry=[Point(lon, lat) for lon, lat in zip(df['longitude'], df['latitude'])],
    crs='EPSG:4326'
)

# Load geology — no buffer(0) on points
geology = gpd.read_file(GEO_DIR / 'triassic_marine.shp')
geology['geometry'] = geology.geometry.buffer(0)  # fix polygon geometry only

# Reproject both to UTM 11N
cands_utm = candidates.to_crs('EPSG:32611')
geo_utm   = geology.to_crs('EPSG:32611')

print(f'Candidates sample:\n{cands_utm.geometry.head(3)}')
print(f'\nGeology bounds: {geo_utm.total_bounds}')
print(f'\nCandidates bounds: {cands_utm.total_bounds}')

# Spatial join using intersects
joined = gpd.sjoin(cands_utm,
                   geo_utm[['ORIG_LABEL','unit_name','rocktype1','geometry']],
                   how='left',
                   predicate='intersects')

print(f'\nCandidates on Triassic: {joined[~joined["ORIG_LABEL"].isna()]["rank"].nunique()}')

# Distance to nearest Triassic polygon for each candidate
geo_union = geo_utm.union_all()
cands_utm['dist_to_triassic_m'] = cands_utm.geometry.apply(
    lambda g: int(g.distance(geo_union))
)

print(f'\nDistance stats:')
print(cands_utm['dist_to_triassic_m'].describe())

Candidates sample:
0    POINT (394450.188 4473421.714)
1    POINT (461493.765 4515237.332)
2      POINT (376663.5 4529490.207)
Name: geometry, dtype: geometry

Geology bounds: [ 392934.5131625  3961817.20539613  764033.15918645 4653145.59172078]

Candidates bounds: [ 331042.00269788 4234744.55605207  482743.19521248 4529490.20690995]

Candidates on Triassic: 0

Distance stats:
count        50.000000
mean      55453.360000
std       60465.250068
min          52.000000
25%        5228.250000
50%       18331.500000
75%      124113.750000
max      166378.000000
Name: dist_to_triassic_m, dtype: float64


In [8]:
# Build final priority table using distance tiers
final = candidates.copy()
final['dist_to_triassic_m'] = cands_utm['dist_to_triassic_m'].values

# Join nearest formation name
def get_nearest_formation(geom, geo_utm):
    distances = geo_utm.geometry.distance(geom)
    idx = distances.idxmin()
    row = geo_utm.loc[idx]
    return row['ORIG_LABEL'], row['unit_name'], row['rocktype1']

print('Finding nearest formation for each candidate...')
results = cands_utm.geometry.apply(lambda g: get_nearest_formation(g, geo_utm))
final['formation_code'] = [r[0] for r in results]
final['formation_name'] = [r[1] for r in results]
final['rock_type']      = [r[2] for r in results]

# Geology bonus based on distance
def geo_bonus(dist):
    if dist <= 100:    return 0.20  # essentially on formation
    elif dist <= 500:  return 0.15
    elif dist <= 2000: return 0.10
    elif dist <= 5000: return 0.05
    else:              return 0.00

final['geo_bonus']       = final['dist_to_triassic_m'].apply(geo_bonus)
final['composite_score'] = (final['ml_probability'] + final['geo_bonus']).clip(0, 1).round(4)
final['priority']        = final['composite_score'].rank(ascending=False, method='min').astype(int)
final = final.sort_values('composite_score', ascending=False).reset_index(drop=True)

# Save
out_path = MODEL_DIR / 'priority_targets.csv'
final[['priority','latitude','longitude','ml_probability','dist_to_triassic_m',
       'formation_code','formation_name','rock_type','geo_bonus','composite_score']]\
    .to_csv(out_path, index=False)

print(f'Saved: {out_path}')
print(f'\n🦴 TOP 20 PALEOWAVE FIELD TARGETS:')
print(final.head(20)[[
    'priority','latitude','longitude','ml_probability',
    'dist_to_triassic_m','formation_code','composite_score'
]].to_string(index=False))

Finding nearest formation for each candidate...
Saved: ..\data\model\priority_targets.csv

🦴 TOP 20 PALEOWAVE FIELD TARGETS:
 priority  latitude   longitude  ml_probability  dist_to_triassic_m formation_code  composite_score
        1 40.404722 -118.243889        0.800458                  52            TRc           1.0000
        2 40.209167 -117.587778        0.795162                 992            TRc           0.8952
        3 40.550278 -118.232500        0.794605                1361            TRc           0.8946
        4 40.434444 -117.686667        0.842316                4781            TRc           0.8923
        5 40.839722 -117.720833        0.792043                1059            TRc           0.8920
        6 40.822778 -117.696944        0.789900                1309            TRc           0.8899
        7 40.418889 -117.704444        0.830465                3338            TRc           0.8805
        8 40.787222 -117.456389        0.853346               13040        

In [9]:
# Save as GeoJSON for QGIS
from shapely.geometry import Point
import geopandas as gpd

final_gdf = gpd.GeoDataFrame(
    final,
    geometry=[Point(lon, lat) for lon, lat in zip(final['longitude'], final['latitude'])],
    crs='EPSG:4326'
)
final_gdf.to_file(MODEL_DIR / 'priority_targets.geojson', driver='GeoJSON')
print('GeoJSON saved.')

GeoJSON saved.
